# Oppimisprojekti 3, osa 2: puheentunnistus Whisper-mallilla

T?ss? notebookissa k?ytet??n Hugging Facen `transformers`-kirjaston Whisper-malleja suomenkielisen puheen tunnistamiseen. Notebook on tehty niin, ett? voit lis?t? omat alle 30 sekunnin ??nitteesi kansioon `audio_samples` ja ajaa vertailun usealla mallikoolla.

**Teko?lyn k?ytt?:** Notebookin runko, analyysikysymykset ja raporttirakenne on laadittu Codex-avustajalla. ??nin?ytteet ja lopullinen arviointi t?ydennet??n omien ajotulosten perusteella.

## Mel-spektrogrammi omin sanoin

Whisper ei sy?t? neuroverkolle raakaa ??niaaltoa. ??ni muunnetaan ensin mel-spektrogrammiksi.

- **x-akseli** kuvaa aikaa: vasemmalta oikealle edet??n ??nitteen alusta loppuun.
- **y-akseli** kuvaa taajuuksia mel-asteikolla. Mel-asteikko painottaa taajuuksia ihmiskuulon kannalta mielekk??mmin kuin lineaarinen hertsiasteikko.
- **v?ri** kuvaa voimakkuutta desibelein?. Kirkkaammat tai voimakkaammat v?rit tarkoittavat, ett? kyseisell? ajanhetkell? ja taajuusalueella on enemm?n energiaa.

T?m? esitysmuoto sopii neuroverkolle paremmin kuin raaka aalto, koska puheen olennaiset piirteet, kuten vokaalit, konsonantit ja ??nenpainot, n?kyv?t paikallisina kuvioina ajan ja taajuuden tasossa. Transformer saa siis j?rjestetyn piirrejonon, jota se voi k?sitell? samaan tapaan kuin osassa 1 tokenijonoa.

In [ ]:
# Tarvittaessa asenna riippuvuudet esimerkiksi:
# %pip install torch transformers librosa soundfile matplotlib pandas jiwer accelerate

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import torch
from IPython.display import Audio, display
from transformers import WhisperProcessor, WhisperForConditionalGeneration

warnings.filterwarnings("ignore")

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Laite:", device)

## ??nin?ytteet

Lis?? omat tiedostot kansioon `audio_samples`. Suositeltu rakenne:

- `selkea_puhe.wav`: selke? suomenkielinen puhe
- `taustahaly.wav`: sama tai vastaava lause taustah?lyll?
- `eri_puhuja.wav`: eri puhujan ??ni
- `murre.wav`: murteellinen puhe
- `english.wav`: lyhyt englanninkielinen puhe

Kirjoita `sample_descriptions`-sanakirjaan kunkin n?ytteen oikea teksti tai arvioitu sis?lt?. Sit? k?ytet??n tarkkuuden arviointiin.

In [ ]:
AUDIO_DIR = Path("audio_samples")
AUDIO_DIR.mkdir(exist_ok=True)

sample_descriptions = {
    # "selkea_puhe.wav": {
    #     "kuvaus": "Selke? suomenkielinen puhe, yksi puhuja, hiljainen huone.",
    #     "oikea_teksti": "T?m? on ensimm?inen kokeilu tunnistaa suomenkielist? puhetta.",
    # },
}

supported = {".wav", ".mp3", ".m4a", ".flac", ".ogg"}
audio_files = sorted([p for p in AUDIO_DIR.iterdir() if p.suffix.lower() in supported])

print(f"L?ytyi {len(audio_files)} ??nitiedostoa kansiosta {AUDIO_DIR.resolve()}")
for p in audio_files:
    print("-", p.name)

In [ ]:
def load_audio(path, target_sr=16000):
    y, sr = librosa.load(path, sr=target_sr, mono=True)
    duration = librosa.get_duration(y=y, sr=sr)
    if duration > 30:
        print(f"Varoitus: {path.name} on {duration:.1f} s. Whisperin peruskokeeseen suositellaan alle 30 s p?tki?.")
    return y, sr, duration

for path in audio_files:
    y, sr, duration = load_audio(path)
    desc = sample_descriptions.get(path.name, {}).get("kuvaus", "Kuvaus puuttuu")
    print(f"
{path.name}: {duration:.2f} s, {sr} Hz, {desc}")
    display(Audio(y, rate=sr))

## Mel-spektrogrammin visualisointi

Aja seuraava solu yhdelle tai useammalle n?ytteelle ja liit? havaintosi raporttiin. Kiinnit? huomiota siihen, miss? kohtaa puhe, tauot ja mahdollinen taustah?ly n?kyv?t.

In [ ]:
def plot_mel_spectrogram(path, n_mels=80):
    y, sr, duration = load_audio(path)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, hop_length=160, n_fft=400)
    mel_db = librosa.power_to_db(mel, ref=np.max)

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=False)
    librosa.display.waveshow(y, sr=sr, ax=axes[0])
    axes[0].set_title(f"Aaltomuoto: {path.name}")

    img = librosa.display.specshow(mel_db, sr=sr, hop_length=160, x_axis="time", y_axis="mel", ax=axes[1])
    axes[1].set_title("Mel-spektrogrammi")
    fig.colorbar(img, ax=axes[1], format="%+2.0f dB")
    plt.tight_layout()
    plt.show()

if audio_files:
    plot_mel_spectrogram(audio_files[0])
else:
    print("Lis?? ??nitiedostoja audio_samples-kansioon ja aja solu uudelleen.")

## Whisper-mallien vertailu

Oletuksena vertaillaan malleja `tiny` ja `base`, koska ne latautuvat ja ajavat kohtuullisen nopeasti. Lis?? listaan esimerkiksi `small` tai `large-v3`, jos k?yt?ss? on tarpeeksi muistia ja aikaa.

In [ ]:
MODEL_SIZES = ["tiny", "base"]  # kokeile my?s: "small", "medium", "large-v3"
LANGUAGE = "fi"
TASK = "transcribe"

def model_id(size):
    return "openai/whisper-large-v3" if size == "large-v3" else f"openai/whisper-{size}"


def load_whisper(size):
    name = model_id(size)
    processor = WhisperProcessor.from_pretrained(name)
    model = WhisperForConditionalGeneration.from_pretrained(name).to(device)
    model.eval()
    return processor, model


def transcribe(path, processor, model, language=LANGUAGE, task=TASK):
    y, sr, duration = load_audio(path)
    inputs = processor(y, sampling_rate=sr, return_tensors="pt")
    input_features = inputs.input_features.to(device)

    start = time.perf_counter()
    with torch.no_grad():
        generated_ids = model.generate(input_features, language=language, task=task)
    seconds = time.perf_counter() - start

    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    return text, duration, seconds

In [ ]:
results = []

for size in MODEL_SIZES:
    print(f"
Ladataan Whisper {size}...")
    processor, whisper_model = load_whisper(size)

    for path in audio_files:
        print(f"  Tunnistetaan: {path.name}")
        prediction, duration, seconds = transcribe(path, processor, whisper_model)
        ref = sample_descriptions.get(path.name, {}).get("oikea_teksti", "")
        results.append({
            "malli": size,
            "tiedosto": path.name,
            "kesto_s": round(duration, 2),
            "aika_s": round(seconds, 2),
            "reaaliaikakerroin": round(seconds / duration, 2) if duration else np.nan,
            "oikea_teksti": ref,
            "tunnistus": prediction,
        })

    del whisper_model
    if device == "cuda":
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
display(results_df)

In [ ]:
# Valinnainen WER-laskenta, jos olet t?ytt?nyt oikeat tekstit.
# %pip install jiwer
try:
    from jiwer import wer
    if not results_df.empty and results_df["oikea_teksti"].str.len().gt(0).any():
        results_df["WER"] = results_df.apply(
            lambda r: wer(r["oikea_teksti"], r["tunnistus"]) if r["oikea_teksti"] else np.nan,
            axis=1,
        )
        display(results_df[["malli", "tiedosto", "aika_s", "reaaliaikakerroin", "WER", "tunnistus"]])
    else:
        print("T?yt? oikea_teksti-arvot sample_descriptions-sanastoon, jos haluat laskea WER-luvun.")
except Exception as exc:
    print("WER-laskentaa ei ajettu:", exc)

## Raportointitaulukko

T?ydenn? ajon j?lkeen tuloksilla.

| ??nin?yte | Kuvaus | tiny: tulos/nopeus | base: tulos/nopeus | small/large: tulos/nopeus | havainto |
|---|---|---|---|---|---|
| selke? puhe | t?yt? | t?yt? | t?yt? | t?yt? | pienikin malli voi riitt?? |
| taustah?ly | t?yt? | t?yt? | t?yt? | t?yt? | h?ly lis?? virheit? |
| eri puhuja | t?yt? | t?yt? | t?yt? | t?yt? | ??nenlaatu ja artikulaatio vaikuttavat |
| murre | t?yt? | t?yt? | t?yt? | t?yt? | murre voi tuottaa normalisointia tai v??rinkuulemista |
| englanti | t?yt? | t?yt? | t?yt? | t?yt? | englanti toimii usein hyvin, koska Whisper on monikielisesti esikoulutettu |

### Analyysi

- Mallin koon kasvaessa tunnistus yleens? paranee, mutta lataus, muistinkulutus ja laskenta-aika kasvavat.
- Pieni malli riitt?? selke??n, lyhyeen ja hyvin ??nitettyyn puheeseen.
- Suurempi malli auttaa ep?selv?ss? puheessa, taustah?lyss?, murteissa ja silloin, kun tarvitaan parempaa oikeinkirjoitusta.
- Meluisa ??ni voi aiheuttaa puuttuvia sanoja, keksittyj? sanoja tai v??r?n kielen/tulkinnan.
- Whisperin mel-spektrogrammi vastaa osan 1 tokenijonoa siin? mieless?, ett? molemmissa transformer k?sittelee j?rjestetty? sy?tejonoa. Osassa 1 sy?te on tekstin tokenisarja, osassa 2 sy?te on ??nen aika-taajuuspiirteiden sarja.

## Vapaaehtoinen Gradio-k?ytt?liittym?

Jos haluat kokeilla mikrofonia selaimessa, poista kommentit ja aja solu. T?m? ei ole pakollinen palautukseen.

In [ ]:
# %pip install gradio
# import gradio as gr
#
# processor, whisper_model = load_whisper("base")
#
# def transcribe_gradio(audio_path):
#     if audio_path is None:
#         return ""
#     text, duration, seconds = transcribe(Path(audio_path), processor, whisper_model)
#     return f"{text}

Kesto: {duration:.1f} s, tunnistusaika: {seconds:.1f} s"
#
# demo = gr.Interface(
#     fn=transcribe_gradio,
#     inputs=gr.Audio(sources=["microphone", "upload"], type="filepath"),
#     outputs=gr.Textbox(lines=6),
#     title="Whisper suomenkieliseen puheentunnistukseen",
# )
# demo.launch()